# Performance Analytics — Manual Build

A simple, from-scratch version of the 8-task mutual fund analysis, written so each
step is easy to read and re-type yourself. Uses **one** benchmark (UTI Nifty 50 Index
Fund) and sticks strictly to the deliverables in the brief — no extra CSV outputs
beyond what was asked for.


## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import os

DATA_DIR = r"C:\Users\HP\bluestock-mf-analytics\data\cleaned"
OUT_DIR = r"C:\Users\HP\bluestock-mf-analytics\reports"
os.makedirs(OUT_DIR, exist_ok=True)

RF = 0.065        # RBI repo rate proxy, annualized
TRADING_DAYS = 252


In [ ]:
nav_long = pd.read_csv(f"{DATA_DIR}/clean_nav.csv", parse_dates=["date"])
perf = pd.read_csv(f"{DATA_DIR}/clean_performance.csv")

# Your NAV data is "long" format (one row per fund per date) — reshape to "wide"
# so each column is a fund and each row is a date. Much easier to work with.
nav_wide = nav_long.pivot(index="date", columns="amfi_code", values="nav").sort_index()

meta_df = perf.drop_duplicates(subset="amfi_code").set_index("amfi_code")[
    ["scheme_name", "fund_house", "category", "expense_ratio_pct"]
]

nav_wide.head()   # check it looks right before continuing


## Task 1 — Daily Returns

**Formula:** `daily_return = nav_t / nav_t-1 - 1`

In [ ]:
returns = nav_wide.pct_change().dropna(how="all")
returns.head()


Validate the distribution — check nothing looks broken (e.g. a 500% single-day move, which would signal bad data):

In [ ]:
summary = pd.DataFrame({
    "mean": returns.mean(),
    "std": returns.std(),
    "min": returns.min(),
    "max": returns.max(),
})
summary["suspect"] = summary[["min","max"]].abs().max(axis=1) > 0.25   # flag >25% single-day moves
summary.sort_values("std", ascending=False).head(10)


## Task 2 — CAGR (1yr, 3yr, 5yr)

**Formula:** `CAGR = (NAV_end / NAV_start) ^ (1/n) - 1`

In [ ]:
def compute_cagr(nav_df, years):
    n_days = years * TRADING_DAYS
    if len(nav_df) < n_days + 1:
        return pd.Series(np.nan, index=nav_df.columns)   # not enough history
    nav_start = nav_df.iloc[-n_days - 1]
    nav_end = nav_df.iloc[-1]
    return (nav_end / nav_start) ** (1/years) - 1

cagr_table = pd.DataFrame({
    "CAGR_1Y": compute_cagr(nav_wide, 1),
    "CAGR_3Y": compute_cagr(nav_wide, 3),
    "CAGR_5Y": compute_cagr(nav_wide, 5),
})
cagr_table = cagr_table.join(meta_df[["scheme_name","category"]])
cagr_table.sort_values("CAGR_3Y", ascending=False)


## Task 3 — Sharpe Ratio

**Formula:** `Sharpe = (Rp - Rf) / Std(Rp) * sqrt(252)`

In [ ]:
rf_daily = RF / TRADING_DAYS
sharpe = (returns.mean() - rf_daily) / returns.std() * np.sqrt(TRADING_DAYS)
sharpe.name = "Sharpe"

sharpe_ranked = sharpe.rank(ascending=False).astype(int)
pd.DataFrame({"Sharpe": sharpe, "Rank": sharpe_ranked}).sort_values("Rank")


## Task 4 — Sortino Ratio

Same idea as Sharpe, but only penalize *downside* volatility (negative days), not all volatility.

In [ ]:
downside_std = returns.where(returns < 0).std()   # keeps only negative returns, then std
sortino = (returns.mean() - rf_daily) / downside_std * np.sqrt(TRADING_DAYS)
sortino.name = "Sortino"
sortino.sort_values(ascending=False)


## Task 5 — Alpha & Beta (OLS regression)

Needs a benchmark return series. Uses the UTI Nifty 50 Index Fund (amfi_code 102885)
already present in the NAV data as a Nifty 50 proxy.

In [ ]:
bench_code = 102885   # UTI Nifty 50 Index Fund
bench_returns = returns[bench_code]
fund_returns = returns.drop(columns=[bench_code])   # everything except the benchmark itself

results = []
for code_ in fund_returns.columns:
    fund_r = fund_returns[code_]
    mask = fund_r.notna() & bench_returns.notna()
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        bench_returns[mask], fund_r[mask]
    )
    results.append((code_, intercept * TRADING_DAYS, slope, r_value**2, p_value))

alpha_beta_df = pd.DataFrame(results, columns=["amfi_code","alpha","beta","r_squared","p_value"]).set_index("amfi_code")
alpha_beta_df = alpha_beta_df.join(meta_df[["scheme_name"]])
alpha_beta_df.sort_values("alpha", ascending=False)


## Task 6 — Maximum Drawdown

**Formula:** `drawdown_t = NAV_t / running_max(NAV) - 1`

In [ ]:
running_max = nav_wide.cummax()
drawdown = nav_wide / running_max - 1
max_dd = drawdown.min()

# Find the peak & trough dates for the worst drawdown
dd_rows = []
for col in nav_wide.columns:
    trough_date = drawdown[col].idxmin()
    peak_date = nav_wide[col].loc[:trough_date].idxmax()
    dd_rows.append((col, max_dd[col], peak_date, trough_date))

dd_df = pd.DataFrame(dd_rows, columns=["amfi_code","max_drawdown","peak_date","trough_date"]).set_index("amfi_code")
dd_df.sort_values("max_drawdown").head(10)   # worst drawdowns first


## Task 7 — Fund Scorecard (0–100)

In [ ]:
scorecard = pd.DataFrame(index=fund_returns.columns)
scorecard["CAGR_3Y"] = cagr_table["CAGR_3Y"]
scorecard["Sharpe"] = sharpe
scorecard["Alpha"] = alpha_beta_df["alpha"]
scorecard["ExpenseRatio"] = meta_df["expense_ratio_pct"]
scorecard["MaxDD"] = dd_df["max_drawdown"]

def pct_rank(s, ascending=True):
    return s.rank(pct=True, ascending=ascending)   # returns a 0-1 percentile rank

r_cagr    = pct_rank(scorecard["CAGR_3Y"])
r_sharpe  = pct_rank(scorecard["Sharpe"])
r_alpha   = pct_rank(scorecard["Alpha"])
r_expense = pct_rank(scorecard["ExpenseRatio"], ascending=False)  # LOWER expense = better, so flip
r_maxdd   = pct_rank(scorecard["MaxDD"], ascending=False)         # closer to 0 = better, so flip

scorecard["score"] = 100 * (0.30*r_cagr + 0.25*r_sharpe + 0.20*r_alpha + 0.15*r_expense + 0.10*r_maxdd)
scorecard = scorecard.join(meta_df[["scheme_name"]]).sort_values("score", ascending=False)
scorecard.head(10)


In [ ]:
# Save deliverables
scorecard.to_csv(f"{OUT_DIR}/fund_scorecard.csv")
alpha_beta_df.to_csv(f"{OUT_DIR}/alpha_beta.csv")
print("Saved fund_scorecard.csv and alpha_beta.csv to", OUT_DIR)


## Task 8 — Benchmark Comparison Chart + Tracking Error

In [ ]:
top5 = scorecard.head(5).index.tolist()

window = min(3*TRADING_DAYS, len(nav_wide)-1)
nav_window = nav_wide.iloc[-window-1:]

fund_rebased = nav_window[top5] / nav_window[top5].iloc[0] * 100
bench_rebased = nav_window[bench_code] / nav_window[bench_code].iloc[0] * 100

fig, ax = plt.subplots(figsize=(12,7))
for code_ in top5:
    ax.plot(fund_rebased.index, fund_rebased[code_], label=meta_df.loc[code_,"scheme_name"])
ax.plot(bench_rebased.index, bench_rebased, label="Nifty 50 Proxy", linewidth=2.5, linestyle="--", color="black")
ax.legend()
ax.set_title("Top 5 Funds vs Nifty 50 Proxy")
plt.tight_layout()
fig.savefig(f"{OUT_DIR}/benchmark_comparison.png", dpi=150)
plt.show()


In [ ]:
# Tracking error — required by the brief, just printed here (no separate CSV needed)
te = (returns[top5].sub(bench_returns, axis=0)).std() * np.sqrt(TRADING_DAYS)
te
